# Environment verification — M0-SETUP-01

Closes the one acceptance criterion of `M0-SETUP-01` that cannot be checked from a
laptop: **`pip install -r requirements.txt` succeeds in a clean Colab cell.**

Run every cell top to bottom on a fresh Colab runtime, then paste the output of
cells 2, 4 and 5 into a new `AGENT_LOG.md` entry.

No project logic lives here — this notebook only reports and calls
(`spec/architecture.md §2`). It is a verification harness, not part of the pipeline.

**What it is actually testing.** The local venvs are empty to start with, so a clean
install there proves very little about Colab, which arrives with its own numpy,
pandas, scipy and sklearn already installed. The real question is whether our lower
bounds force Colab to *downgrade* one of those, or to build something from source
because no wheel matches. Cell 4 watches for exactly that.

In [1]:
# 2 — what interpreter are we on?
import sys, platform

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    import google.colab  # noqa: F401
    print("Runtime : Google Colab")
except ImportError:
    print("Runtime : not Colab (local kernel or other host)")

# spec/architecture.md section 1 pins 3.11. sinter 1.16 needs >= 3.12, so the
# interpreter decides whether we resolve sinter 1.15 or 1.16. Record it.
print()
print("NOTE: record this Python version - it determines the resolved sinter version.")

Python : 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Runtime : Google Colab

NOTE: record this Python version - it determines the resolved sinter version.


In [4]:
# 3 — find the repo on mounted Drive, without upload() fallback
import os
import pathlib

def find_requirements_txt():
    # optional manual override for a nonstandard location
    manual = os.environ.get("QECSCREEN_REPO")
    if manual:
        repo = pathlib.Path(manual).expanduser()
        req = repo / "requirements.txt"
        if req.is_file():
            return req.resolve()

    # common Drive mount locations
    roots = []
    for base in [
        pathlib.Path("/content"),
        pathlib.Path("/content/drive"),
        pathlib.Path("/content/drive/MyDrive"),
        pathlib.Path("/content/drive/Shareddrives"),
    ]:
        if base.exists():
            roots.append(base)

    # Explicitly check the exact nested path used in your Drive layout:
    # /content/drive/MyDrive/qecscreen/qecscreen/requirements.txt
    search_roots = []
    for root in roots:
        search_roots.append(root)
        search_roots.append(root / "qecscreen")
        search_roots.append(root / "qecscreen" / "qecscreen")

    # First: direct repo candidates
    for root in search_roots:
        req = root / "requirements.txt"
        if req.is_file():
            return req.resolve()

    # Second: repo dir named qecscreen
    for root in search_roots:
        repo_dir = root
        if repo_dir.name.lower() == "qecscreen":
            req = repo_dir / "requirements.txt"
            if req.is_file():
                return req.resolve()

    # Third: fallback search under mounted locations
    for root in roots:
        for candidate in root.rglob("requirements.txt"):
            if "qecscreen" in str(candidate).lower():
                return candidate.resolve()

    return None

# Mount Drive if it is not already mounted
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

REQ = find_requirements_txt()

if REQ is None:
    raise FileNotFoundError(
        "requirements.txt not found in the mounted Drive. "
        "Expected a repo layout like:\n"
        "  /content/drive/MyDrive/qecscreen/qecscreen/requirements.txt\n"
        "or /content/drive/Shareddrives/<name>/qecscreen/requirements.txt\n"
        "Then rerun this cell. Do not use files.upload() here; it is not reliable "
        "in a VS Code-connected Colab session."
    )

REPO = REQ.parent.resolve()
print("Using repo root:", REPO)
print("Using requirements:", REQ)
print()
print(REQ.read_text(encoding="utf-8"))

Mounted at /content/drive
Using repo root: /content/drive/MyDrive/qecscreen/qecscreen
Using requirements: /content/drive/MyDrive/qecscreen/qecscreen/requirements.txt

# Pinned per spec/architecture.md section 1.
# Must install in a single Colab/Kaggle cell with no compiler and no API key.
#
# Interpreter: Python 3.11 (spec/architecture.md section 1). Verified to install
# from wheels only and pass pytest on 3.11.9 and on 3.13.7.
#
# Bound style: lower bounds for the ambient scientific stack, because Kaggle and
# Colab ship their own numpy/pandas/scipy and an exact pin forces a downgrade
# that breaks the one-cell install (AGENTS.md section 3). An upper bound is
# added only where there is a stated reason to expect a breaking major release.

numpy>=1.26,<3
scipy>=1.12
pandas>=2.2
pyarrow>=15
networkx>=3.2

# Label-determining. These three versions enter protocol_hash (CONTRACT.md
# INV-6), so a major release that changes decoding or sampling behaviour would
# silently invalidate every l

In [5]:
# 4 — the actual acceptance criterion
import subprocess, sys

if REQ is None or REPO is None:
    raise RuntimeError("Repo discovery failed. Stop before pip install.")

proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REQ)],
    capture_output=True,
    text=True,
)
print("pip exit code:", proc.returncode)
print()

WATCH = (
    "Successfully installed",
    "Attempting uninstall",
    "Successfully uninstalled",
    "Building wheel",
    "Running setup.py",
    "ERROR",
    "incompatible",
)
for line in (proc.stdout + proc.stderr).splitlines():
    if any(token in line for token in WATCH):
        print(line)

assert proc.returncode == 0, "pip install FAILED - paste the full output into AGENT_LOG.md"

pip exit code: 0



In [6]:
# 5 — what actually resolved
import importlib
from importlib.metadata import version, PackageNotFoundError

importlib.invalidate_caches()

PINNED = [
    "numpy", "scipy", "pandas", "pyarrow", "networkx",
    "stim", "sinter", "ldpc",
    "scikit-learn", "lightgbm", "pytest",
    "matplotlib", "pymatching",
]
for name in PINNED:
    try:
        print(name.ljust(14), version(name))
    except PackageNotFoundError:
        print(name.ljust(14), "MISSING")

numpy          2.1.3
scipy          1.16.3
pandas         2.2.3
pyarrow        18.1.0
networkx       3.6.1
stim           1.16.0
sinter         1.16.0
ldpc           2.4.1
scikit-learn   1.6.1
lightgbm       4.6.0
pytest         8.4.2
matplotlib     3.10.0
pymatching     2.4.0


In [10]:
!pip install "pyparsing<3.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: pyparsing
    Found existing installation: pyparsing 3.3.2
    Uninstalling pyparsing-3.3.2:
      Successfully uninstalled pyparsing-3.3.2


In [11]:
!pip install -U matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 93.9 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0


In [12]:
import subprocess, sys

check = subprocess.run(
    [
        sys.executable, "-W", "error", "-c",
        "import stim, sinter, ldpc, networkx, pandas, numpy, scipy, sklearn, lightgbm, pyarrow; "
        "from ldpc import BpOsdDecoder; "
        "from ldpc.sinter_decoders import SinterBpOsdDecoder; "
        "print('imports clean under -W error')"
    ],
    capture_output=True,
    text=True,
)
print(check.stdout)
print(check.stderr)
print("returncode:", check.returncode)

imports clean under -W error


returncode: 0


In [13]:
# 7 — do the pinned CONTRACT decoder parameters work on this runtime?
# Parameters are read from protocol.py, never retyped here (INV-6 / no spec drift).
import sys

src = REPO / "src"
if not src.is_dir():
    print("src/ is not next to requirements.txt - skipping.")
    print("This cell only runs where the repo source is reachable.")
else:
    sys.path.insert(0, str(src))
    import numpy as np
    from ldpc import BpOsdDecoder
    from qecscreen.protocol import DECODER_PARAMS

    kwargs = {k: v for k, v in DECODER_PARAMS.items() if k not in ("library", "decoder")}
    H = np.array([[1, 1, 0], [0, 1, 1]], dtype=np.uint8)
    dec = BpOsdDecoder(H, error_rate=0.01, **kwargs)
    print("BpOsdDecoder accepted:", kwargs)
    print("osd_method as the library stores it:", dec.osd_method)

BpOsdDecoder accepted: {'bp_method': 'minimum_sum', 'max_iter': 30, 'ms_scaling_factor': 0.625, 'osd_method': 'osd_cs', 'osd_order': 10}
osd_method as the library stores it: OSD_CS


In [14]:
# 8 — optional: the full suite, if the repo source is reachable
import subprocess, sys

if not (REPO / "tests").is_dir():
    print("tests/ not reachable - skipping. Cells 4 and 5 are the acceptance criterion.")
else:
    run = subprocess.run(
        [sys.executable, "-m", "pytest"],
        cwd=str(REPO), capture_output=True, text=True,
    )
    print(run.stdout[-3000:])
    print(run.stderr[-2000:])
    print("pytest exit code:", run.returncode)

.......................                                                  [100%]
23 passed in 5.16s


pytest exit code: 0


## Recording the result

Paste the output of **cells 2, 4 and 5** into a new `AGENT_LOG.md` entry.

The criterion is met when:

- cell 4 prints `pip exit code: 0`, **and**
- no `Attempting uninstall` line names `numpy`, `pandas`, `scipy` or `scikit-learn`
  — a downgrade of Colab's preinstalled stack means our lower bounds are fighting
  the host, **and**
- no `Building wheel` / `Running setup.py` line appears — that would mean a
  from-source build, which needs a compiler and breaks the one-cell install.

If any of those three fails, that is a real finding: report it rather than
working around it, and do not loosen a bound without recording why.